# **● Task2 — Data Engineering**

### **Import Libraries**

In [1]:
from datasets import load_dataset

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    lower,
    regexp_replace,
    trim,
    concat_ws,
    length,
    when,
    sum
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    Tokenizer,
    StopWordsRemover,
    HashingTF,
    IDF
)

import time

### **Start Spark**

In [2]:
SEED = 42

spark = (
    SparkSession.builder
    .appName("Task2_Data_Engineering")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

## **Load Dataset**

In [3]:
dataset = load_dataset(
    "bagadbilla/amazon-reviews-2023-trimmed",
    data_dir="Beauty_and_Personal_Care",
    split="train"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.66k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/95 [00:00<?, ?it/s]

Beauty_and_Personal_Care/00000.parquet:   0%|          | 0.00/49.0M [00:00<?, ?B/s]

Beauty_and_Personal_Care/00001.parquet:   0%|          | 0.00/37.9M [00:00<?, ?B/s]

...

Beauty_and_Personal_Care/00091.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

Beauty_and_Personal_Care/00092.parquet:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Beauty_and_Personal_Care/00093.parquet:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

Beauty_and_Personal_Care/00094.parquet:   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
SAMPLE_SIZE = 500000

sample_dataset = dataset.select(range(SAMPLE_SIZE))

spark_df = spark.createDataFrame(sample_dataset)

In [5]:
spark_df = spark_df.repartition(8)

print("Number of Partitions:")
print(spark_df.rdd.getNumPartitions())

Number of Partitions:
8


In [6]:
spark_df.cache()

spark_df.count()

print("Dataset Cached Successfully")

Dataset Cached Successfully


In [7]:
spark_df.show(5, truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### **Data Cleanning**

In [8]:
spark_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in spark_df.columns
]).show()

+------+----+-----+
|rating|text|title|
+------+----+-----+
|     0|   0|    0|
+------+----+-----+



In [9]:
spark_df = spark_df.dropna()

In [10]:
spark_df = spark_df.dropDuplicates()

print("Duplicate removal completed.")

Duplicate removal completed.


In [11]:
spark_df = spark_df.withColumn(
    "review",
    concat_ws(" ", col("title"), col("text"))
)

In [12]:
spark_df = (
    spark_df
    .withColumn("review", lower(col("review")))
    .withColumn("review", regexp_replace("review", "[^a-zA-Z ]", " "))
    .withColumn("review", trim(col("review")))
)

In [13]:
spark_df = spark_df.filter(
    length(col("review")) > 0
)

In [14]:
spark_df.select(
    "rating",
    "review"
).show(5, truncate=False)

+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|rating|review                                                                                                                                                                                                                                                                                                                

In [15]:
label_indexer = StringIndexer(
    inputCol="rating",
    outputCol="label"
)

### **Tokenize the Review Text**

In [16]:
tokenizer = Tokenizer(
    inputCol="review",
    outputCol="words"
)

### **Remove Stop Words**

In [17]:
remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

In [18]:
hashingTF = HashingTF(
    inputCol="filtered_words",
    outputCol="rawFeatures",
    numFeatures=10000
)

### **Apply TF-IDF**

In [19]:
idf = IDF(
    inputCol="rawFeatures",
    outputCol="features"
)

### **Construct the PySpark Pipeline**

In [20]:
pipeline = Pipeline(
    stages=[
        label_indexer,
        tokenizer,
        remover,
        hashingTF,
        idf
    ]
)

print("Pipeline Created Successfully")

Pipeline Created Successfully


In [21]:
start = time.time()

pipeline_model = pipeline.fit(spark_df)

end = time.time()

print(f"Training Time : {end-start:.2f} seconds")

Training Time : 64.96 seconds


## **Transform the Dataset**

In [22]:

processed_df = pipeline_model.transform(spark_df)

print("Feature engineering completed.")

Feature engineering completed.


In [23]:
processed_df.select(
    "label",
    "review",
    "features"
).show(5, truncate=False)

+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### **Split the Dataset into Training and Testing Sets**

In [24]:
train_df, test_df = processed_df.randomSplit(
    [0.8, 0.2],
    seed=SEED
)

In [25]:
print("Training Rows :", train_df.count())

print("Testing Rows :", test_df.count())

Training Rows : 384795
Testing Rows : 96509


### **Save the Pipeline**

In [26]:
from google.colab import drive
import os

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Create folder
SAVE_PATH = "/content/drive/MyDrive/7006SCN"
os.makedirs(SAVE_PATH, exist_ok=True)

# Step 4: Save the pipeline
pipeline_model.write() \
    .overwrite() \
    .save(f"{SAVE_PATH}/Pipeline_Task2")

Mounted at /content/drive


In [27]:
# Step 3: Save the processed dataset
processed_df.write.mode("overwrite").parquet(
    f"{SAVE_PATH}/processed_reviews"
)

print("Everything is permanently saved to Google Drive.")

Everything is permanently saved to Google Drive.


In [28]:
processed_df.printSchema()

root
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)
 |-- review: string (nullable = false)
 |-- label: double (nullable = false)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)



In [29]:
processed_df.select(
    "rating",
    "label",
    "words",
    "filtered_words",
    "features"
).show(5, truncate=False)

+------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------